# Simulacion de salidas de evacuacion optimizadas

---

**Cómo se generan las paredes**

El edificio se construye en tres capas dentro de `make_building`:

Primero se ponen las paredes exteriores — simplemente se asigna `WALL` a toda la primera fila, última fila, primera columna y última columna. Es el perímetro cerrado.

Después se añaden dos paredes internas horizontales en las filas `n//3` y `2*n//3` (un tercio y dos tercios del edificio), simulando divisiones de planta. Estas paredes no son sólidas — se abren 3 huecos aleatorios en cada una usando la semilla de obstáculos, que son los pasillos entre zonas.

Finalmente se colocan obstáculos pequeños (mesas, columnas) en posiciones aleatorias del interior. Cada obstáculo es un rectángulo de 1×1 a 2×2 celdas. El slider **Semilla obstáculos** controla exactamente qué planta se genera — misma semilla siempre produce el mismo edificio.

---


**El valor de fitness y el costo de salidas**

La fórmula es:

```
fitness = −tiempo_promedio − λ × n_salidas
```

El fitness es siempre negativo porque el objetivo es **minimizar** el tiempo, y para expresarlo como maximización se niega, es decir que llegar a 0 significa que todas las personas inician afuera. Ejemplos concretos con λ=2:

- `fitness = −45` → el AG tardó 45 pasos en evacuar con 0 salidas extra de penalización, o 40 pasos con 2-3 salidas penalizadas
- `fitness = −20` → evacuación muy rápida, buena configuración
- `fitness = −200` → casi nadie evacuó o hay demasiadas salidas muy caras

El término `λ × n_salidas` representa el **costo de instalar cada salida** — en la realidad cada salida de emergencia tiene costo real: construcción, señalización, mantenimiento, inspecciones. Sin esta penalización el AG pondría salidas en todo el perímetro, lo cual no es realista.

El efecto de λ en la solución es directo:

| λ | Comportamiento del AG |
|---|---|
| 0.0 | Ignora el costo, pone salidas en todo el borde |
| 2.0 | Balance moderado, típicamente 3-6 salidas bien ubicadas |
| 5.0 | Muy conservador, busca 1-2 salidas perfectamente posicionadas |
| 10.0 | Casi nunca abre salidas, el costo domina completamente |

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from collections import deque
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ─── Estados de la grilla ─────────────────────────────────────────────────────
EMPTY   = 0   # pasillo libre
WALL    = 1   # pared / obstáculo
PERSON  = 2   # persona evacuando
EXIT    = 3   # salida de emergencia

CMAP = mcolors.ListedColormap(['#f5f0e8', '#4a4a4a', '#3a86ff', '#00b050', '#ffffff'])
NORM = mcolors.BoundaryNorm([0, 1, 2, 3, 4, 5], CMAP.N)


def _border_cells(n):
    """Lista ordenada de (r,c) de todas las celdas del borde exterior."""
    cells = []
    for c in range(n):           cells.append((0, c))
    for r in range(1, n):        cells.append((r, n-1))
    for c in range(n-2, -1, -1): cells.append((n-1, c))
    for r in range(n-2, 0, -1):  cells.append((r, 0))
    return cells


def make_building(n, obstacle_seed=0):
    """
    Genera la planta del edificio con paredes exteriores,
    paredes internas con huecos y obstáculos pequeños.
    Devuelve la grilla base SIN salidas ni personas.
    """
    rng  = np.random.RandomState(obstacle_seed)
    grid = np.zeros((n, n), dtype=np.int8)

    # Paredes exteriores
    grid[0, :]  = WALL
    grid[-1, :] = WALL
    grid[:, 0]  = WALL
    grid[:, -1] = WALL

    # Paredes internas horizontales con huecos (pasillos)
    for row in [n // 3, 2 * n // 3]:
        grid[row, 1:-1] = WALL
        gaps = rng.choice(range(1, n - 1), size=3, replace=False)
        for g in gaps:
            grid[row, g] = EMPTY

    # Obstáculos pequeños (mesas/columnas)
    for _ in range(n // 5):
        r = rng.randint(2, n - 2)
        c = rng.randint(2, n - 2)
        if grid[r, c] == EMPTY:
            h = rng.randint(1, 3)
            w = rng.randint(1, 3)
            grid[r:r+h, c:c+w] = WALL

    return grid


def place_exits(building, exit_mask):
    """Coloca salidas en el borde según exit_mask (vector binario = perímetro)."""
    grid   = building.copy()
    border = _border_cells(grid.shape[0])
    for idx, (r, c) in enumerate(border):
        if exit_mask[idx] == 1:
            grid[r, c] = EXIT
    return grid


def place_people(grid, n_people, seed=0):
    """Distribuye n_people personas en celdas EMPTY del interior."""
    rng  = np.random.RandomState(seed)
    g    = grid.copy()
    n    = g.shape[0]
    free = [(r, c) for r, c in zip(*np.where(g == EMPTY))
            if 1 < r < n-1 and 1 < c < n-1]
    chosen = rng.choice(len(free), size=min(n_people, len(free)), replace=False)
    for i in chosen:
        r, c = free[i]
        g[r, c] = PERSON
    return g

In [3]:
# ─── Autómata celular de evacuación ──────────────────────────────────────────

def compute_floor_field(grid):
    """
    BFS desde todas las salidas para calcular distancia mínima de cada celda
    a la salida más cercana (campo de piso estático).
    """
    n    = grid.shape[0]
    dist = np.full((n, n), np.inf)
    from collections import deque
    queue = deque()

    for r in range(n):
        for c in range(n):
            if grid[r, c] == EXIT:
                dist[r, c] = 0
                queue.append((r, c))

    while queue:
        r, c = queue.popleft()
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n:
                if grid[nr, nc] != WALL and dist[nr, nc] == np.inf:
                    dist[nr, nc] = dist[r, c] + 1
                    queue.append((nr, nc))
    return dist


def evacuation_step(grid, floor_field):
    """
    Avanza un paso de evacuación:
    - Cada persona se mueve al vecino (8 dir) con menor distancia a salida.
    - Conflictos (dos personas, una celda) se resuelven aleatoriamente.
    - Personas que alcanzan EXIT son eliminadas (escaparon).
    Devuelve (nueva_grilla, n_escapados).
    """
    n        = grid.shape[0]
    new_grid = grid.copy()
    people   = list(zip(*np.where(grid == PERSON)))
    np.random.shuffle(people)

    claimed = {}
    for r, c in people:
        best_dist = floor_field[r, c]
        best_cell = (r, c)

        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r + dr, c + dc
                if not (0 <= nr < n and 0 <= nc < n):
                    continue
                ct = grid[nr, nc]
                if ct == WALL:
                    continue
                if ct == EXIT:
                    best_cell = (nr, nc)
                    best_dist = -1
                    break
                if floor_field[nr, nc] < best_dist and ct != PERSON:
                    best_dist = floor_field[nr, nc]
                    best_cell = (nr, nc)
            if best_dist == -1:
                break

        if best_cell not in claimed:
            claimed[best_cell] = []
        claimed[best_cell].append((r, c))

    escaped = 0
    for dest, origins in claimed.items():
        winner = origins[np.random.randint(len(origins))]
        all_origins = {o: (o == winner) for o in origins}
        for origin, wins in all_origins.items():
            r, c = origin
            if wins:
                nr, nc = dest
                if grid[nr, nc] == EXIT:
                    new_grid[r, c] = EMPTY
                    escaped += 1
                elif origin != dest:
                    new_grid[r, c]   = EMPTY
                    new_grid[nr, nc] = PERSON
            # losers stay

    # Restaurar salidas
    for r in range(n):
        for c in range(n):
            if grid[r, c] == EXIT:
                new_grid[r, c] = EXIT

    return new_grid, escaped


def simulate_evacuation(grid, max_steps=200):
    """
    Simula hasta que todos evacuen o se agoten los pasos.
    Devuelve (lista_de_estados, pasos_totales).
    """
    states        = [grid.copy()]
    total_escaped = 0
    total_people  = int(np.sum(grid == PERSON))
    floor_field   = compute_floor_field(grid)

    for step in range(max_steps):
        new_grid, esc = evacuation_step(states[-1], floor_field)
        total_escaped += esc
        states.append(new_grid)
        if total_escaped >= total_people:
            return states, step + 1

    return states, max_steps

In [4]:
# ─── Fitness + Algoritmo Genético ────────────────────────────────────────────
# Cromosoma: vector binario de longitud = perímetro del edificio.
# bit=1 → esa celda del borde es una salida de emergencia.

def fitness_evacuation(exit_mask, building, n_people, lam=2.0,
                       max_steps=200, n_trials=3, people_seed=0):
    """
    fitness = -tiempo_promedio_evacuación - λ * n_salidas

    Negativo porque minimizamos tiempo. Menos negativo = mejor.
    λ penaliza el número de salidas (costo de instalación).
    Sin salidas → penalización máxima.
    """
    n_exits = int(exit_mask.sum())
    if n_exits == 0:
        return float(-max_steps * 2)

    grid_exits = place_exits(building, exit_mask)
    times = []
    for trial in range(n_trials):
        grid = place_people(grid_exits, n_people, seed=people_seed + trial)
        _, t = simulate_evacuation(grid, max_steps)
        times.append(t)

    return float(-np.mean(times) - lam * n_exits)


def _tournament(population, fitnesses, k=3):
    idx = np.random.choice(len(population), k, replace=False)
    return population[idx[np.argmax([fitnesses[i] for i in idx])]].copy()

def _crossover(p1, p2):
    mask = np.random.rand(len(p1)) < 0.5
    return np.where(mask, p1, p2).astype(np.uint8)

def _mutate(ind, rate):
    return np.bitwise_xor(ind, (np.random.rand(len(ind)) < rate).astype(np.uint8))


def genetic_algorithm_evacuation(
    building,
    n_people      = 50,
    lam           = 2.0,
    pop_size      = 40,
    generations   = 40,
    mutation_rate = 0.03,
    elitism       = 2,
    max_steps     = 200,
    seed          = 42,
    callback      = None,
):
    if seed is not None:
        np.random.seed(seed)

    chrom_len  = len(_border_cells(building.shape[0]))
    population = [(np.random.rand(chrom_len) < 0.05).astype(np.uint8)
                  for _ in range(pop_size)]
    history    = {'best': [], 'avg': [], 'best_individuals': []}

    for gen in range(generations):
        fits     = [fitness_evacuation(ind, building, n_people, lam, max_steps)
                    for ind in population]
        best_idx = int(np.argmax(fits))
        history['best'].append(fits[best_idx])
        history['avg'].append(float(np.mean(fits)))
        history['best_individuals'].append(population[best_idx].copy())

        if callback:
            callback(gen, fits[best_idx], float(np.mean(fits)))

        sorted_idx = np.argsort(fits)[::-1]
        new_pop    = [population[i].copy() for i in sorted_idx[:elitism]]
        while len(new_pop) < pop_size:
            child = _crossover(_tournament(population, fits),
                               _tournament(population, fits))
            new_pop.append(_mutate(child, mutation_rate))
        population = new_pop

    return history

In [ ]:
# ─── Visualización y Dashboard ───────────────────────────────────────────────

def plot_comparison(best_mask, building, n_people, history,
                    max_steps=200, people_seed=0):
    n      = building.shape[0]
    border = _border_cells(n)

    # Naive: salidas en las 4 esquinas
    naive_mask = np.zeros(len(border), dtype=np.uint8)
    for ci in [1, n-2, 2*n-1, min(3*n-1, len(border)-1)]:
        naive_mask[ci] = 1

    g_naive = place_people(place_exits(building, naive_mask), n_people, seed=people_seed)
    g_opt   = place_people(place_exits(building, best_mask),  n_people, seed=people_seed)
    _, t_naive = simulate_evacuation(g_naive, max_steps)
    _, t_opt   = simulate_evacuation(g_opt,   max_steps)

    ff_opt = compute_floor_field(place_exits(building, best_mask))
    ff_opt[ff_opt == np.inf] = np.nan

    legend = [
        mpatches.Patch(color='#f5f0e8', label='Pasillo'),
        mpatches.Patch(color='#4a4a4a', label='Pared'),
        mpatches.Patch(color='#3a86ff', label='Persona'),
        mpatches.Patch(color='#00b050', label='Salida'),
    ]

    fig = plt.figure(figsize=(15, 9))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.3)

    ax0 = fig.add_subplot(gs[0, 0])
    ax0.imshow(g_naive, cmap=CMAP, norm=NORM, interpolation='nearest')
    ax0.set_title(f'Salidas en esquinas (naive)\n{int(naive_mask.sum())} salidas', fontsize=10)
    ax0.axis('off')

    ax1 = fig.add_subplot(gs[0, 1])
    ax1.imshow(g_opt, cmap=CMAP, norm=NORM, interpolation='nearest')
    ax1.set_title(f'Salidas optimizadas (AG)\n{int(best_mask.sum())} salidas', fontsize=10)
    ax1.axis('off')

    ax2 = fig.add_subplot(gs[0, 2])
    im  = ax2.imshow(ff_opt, cmap='RdYlGn_r', interpolation='nearest')
    plt.colorbar(im, ax=ax2, fraction=0.046, label='Pasos hasta salida')
    ax2.set_title('Campo de piso (AG)\nVerde=cerca, Rojo=lejos', fontsize=10)
    ax2.axis('off')

    ax3 = fig.add_subplot(gs[1, 0])
    bars = ax3.bar(['Naive', 'AG'], [t_naive, t_opt],
                   color=['#e63946', '#2a9d8f'], width=0.5, edgecolor='white')
    for bar, val in zip(bars, [t_naive, t_opt]):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Pasos para evacuar')
    ax3.set_title('Tiempo de evacuación', fontsize=10)
    ax3.set_ylim(0, max(t_naive, t_opt) * 1.25)
    ax3.spines[['top', 'right']].set_visible(False)

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.imshow(building, cmap=CMAP, norm=NORM, interpolation='nearest', alpha=0.3)
    for idx, (r, c) in enumerate(border):
        if best_mask[idx] == 1:
            ax4.plot(c, r, 's', color='#00b050', markersize=9, markeredgecolor='white')
        if naive_mask[idx] == 1:
            ax4.plot(c, r, '^', color='#e63946', markersize=7, markeredgecolor='white')
    ax4.set_title('Posición de salidas\n■ Verde=AG   ▲ Rojo=Naive', fontsize=10)
    ax4.axis('off')
    ax4.legend(handles=legend, loc='lower right', fontsize=7)

    ax5 = fig.add_subplot(gs[1, 2])
    gens = range(len(history['best']))
    ax5.plot(gens, history['best'], color='#2a9d8f', lw=2,   label='Mejor')
    ax5.plot(gens, history['avg'],  color='#8ecae6', lw=1.5, label='Promedio', ls='--')
    ax5.fill_between(gens, history['avg'], history['best'], alpha=0.15, color='#2a9d8f')
    ax5.set_xlabel('Generación')
    ax5.set_ylabel('Fitness (−tiempo − λ·salidas)')
    ax5.set_title('Convergencia del AG', fontsize=10)
    ax5.legend(fontsize=9)
    ax5.grid(True, lw=0.4, alpha=0.5)
    ax5.spines[['top', 'right']].set_visible(False)

    imp = (t_naive - t_opt) / t_naive * 100 if t_naive > 0 else 0
    fig.suptitle(
        f'Optimización de salidas  —  Mejora: {imp:.1f}%  |  '
        f'Naive: {t_naive} pasos  →  AG: {t_opt} pasos',
        fontsize=12, fontweight='bold'
    )
    plt.show()


def animate_evacuation(best_mask, building, n_people, max_steps=200, people_seed=0):
    n      = building.shape[0]
    border = _border_cells(n)
    naive_mask = np.zeros(len(border), dtype=np.uint8)
    for ci in [0, n-1, 2*n-1, min(3*n-1, len(border)-1)]:
        naive_mask[ci] = 1

    g_naive = place_people(place_exits(building, naive_mask), n_people, seed=people_seed)
    g_opt   = place_people(place_exits(building, best_mask),  n_people, seed=people_seed)
    st_n, _ = simulate_evacuation(g_naive, max_steps)
    st_o, _ = simulate_evacuation(g_opt,   max_steps)

    max_t = max(len(st_n), len(st_o))
    while len(st_n) < max_t: st_n.append(st_n[-1])
    while len(st_o) < max_t: st_o.append(st_o[-1])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
    im1 = ax1.imshow(st_n[0], cmap=CMAP, norm=NORM, interpolation='nearest')
    im2 = ax2.imshow(st_o[0], cmap=CMAP, norm=NORM, interpolation='nearest')
    t1  = ax1.set_title('Naive  t=0')
    t2  = ax2.set_title('AG optimizado  t=0')
    ax1.axis('off'); ax2.axis('off')

    def update(frame):
        im1.set_data(st_n[frame]); im2.set_data(st_o[frame])
        p1 = int(np.sum(st_n[frame] == PERSON))
        p2 = int(np.sum(st_o[frame] == PERSON))
        t1.set_text(f'Naive  t={frame}  restantes={p1}')
        t2.set_text(f'AG  t={frame}  restantes={p2}')
        return im1, im2, t1, t2

    ani = animation.FuncAnimation(fig, update, frames=max_t,
                                   interval=100, blit=True, repeat=False)
    plt.tight_layout(); plt.close(fig)
    return ani


# ─── Widgets ─────────────────────────────────────────────────────────────────
w_grid   = widgets.IntSlider(value=25, min=15, max=40, step=5,
                             description='Grilla N×N:', style={'description_width':'120px'},
                             layout=widgets.Layout(width='350px'))
w_people = widgets.IntSlider(value=50, min=10, max=150, step=10,
                             description='Personas:', style={'description_width':'120px'},
                             layout=widgets.Layout(width='350px'))
w_lam    = widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5,
                               description='λ costo salida:', readout_format='.1f',
                               style={'description_width':'120px'},
                               layout=widgets.Layout(width='350px'))
w_obs    = widgets.IntSlider(value=0, min=0, max=99, step=1,
                             description='Semilla obstáculos:', style={'description_width':'120px'},
                             layout=widgets.Layout(width='350px'))
w_pop    = widgets.IntSlider(value=40, min=20, max=100, step=10,
                             description='Población AG:', style={'description_width':'120px'},
                             layout=widgets.Layout(width='350px'))
w_gens   = widgets.IntSlider(value=40, min=10, max=100, step=10,
                             description='Generaciones:', style={'description_width':'120px'},
                             layout=widgets.Layout(width='350px'))
w_mut    = widgets.FloatSlider(value=0.03, min=0.005, max=0.15, step=0.005,
                               description='Mutación:', readout_format='.3f',
                               style={'description_width':'120px'},
                               layout=widgets.Layout(width='350px'))
w_seed   = widgets.IntText(value=42, description='Semilla AG:',
                           style={'description_width':'120px'},
                           layout=widgets.Layout(width='220px'))
w_run    = widgets.Button(description='▶  Ejecutar AG', button_style='success',
                          layout=widgets.Layout(width='160px', height='36px'))
w_anim   = widgets.Button(description='🎬  Animar', button_style='info',
                          layout=widgets.Layout(width='140px', height='36px'),
                          disabled=True)
w_prog   = widgets.IntProgress(value=0, min=0, max=100, bar_style='success',
                                layout=widgets.Layout(width='100%'))
w_status = widgets.Label(value='Configura los parámetros y presiona Ejecutar.')
w_out    = widgets.Output()

_state = {'history': None, 'building': None, 'params': {}}


def on_run(_):
    w_run.disabled = True; w_anim.disabled = True; w_prog.value = 0
    total_gens = w_gens.value
    building   = make_building(w_grid.value, obstacle_seed=w_obs.value)
    _state['building'] = building

    def cb(gen, best, avg):
        w_prog.value   = int((gen+1)/total_gens*100)
        w_status.value = f'Gen {gen+1}/{total_gens}  ·  Mejor: {best:.1f}  ·  Avg: {avg:.1f}'

    with w_out:
        clear_output(wait=True); print('Ejecutando AG...')

    params = dict(building=building, n_people=w_people.value, lam=w_lam.value,
                  pop_size=w_pop.value, generations=total_gens,
                  mutation_rate=w_mut.value, seed=w_seed.value, callback=cb)
    _state['params'] = params

    history = genetic_algorithm_evacuation(**params)
    _state['history'] = history
    best_mask = history['best_individuals'][-1]

    with w_out:
        clear_output(wait=True)
        plot_comparison(best_mask, building, w_people.value, history)

    w_status.value = f'✓ Listo. Mejor fitness: {history["best"][-1]:.1f}'
    w_run.disabled = False; w_anim.disabled = False


def on_animate(_):
    if _state['history'] is None: return
    best_mask = _state['history']['best_individuals'][-1]
    with w_out:
        clear_output(wait=True)
        w_status.value = 'Generando animación...'
        ani = animate_evacuation(best_mask, _state['building'],
                                 _state['params']['n_people'])
        _state['_ani'] = ani
        display(HTML(ani.to_jshtml()))
        w_status.value = '✓ Animación lista.'


w_run.on_click(on_run); w_anim.on_click(on_animate)

col1 = widgets.VBox([w_grid, w_people, w_lam, w_obs])
col2 = widgets.VBox([w_pop, w_gens, w_mut, w_seed, widgets.HBox([w_run, w_anim])])
display(widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 8px">🚪 Optimización de salidas de emergencia con AG</h3>'),
    widgets.HBox([col1, col2], layout=widgets.Layout(gap='32px')),
    w_prog, w_status, w_out,
]))